# L13 · DAPO와 안정적인 reasoning RL

## Goal

- DAPO 네 요소를 독립시킨다
- Dr. GRPO reduction을 비교한다
- GSPO sequence ratio를 구분한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L13:toy:42").hexdigest()
print(f"lesson=L13 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L13 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:1d3b156ef3789b22828f7a1491201aead31d8baa5d694383347d979efd7b9fa8 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: GRPO → **DAPO·Dr.GRPO·GSPO** → 안정성 평가

$$L_{DAPO}=L_{clip\text{-}higher}+L_{dynamic\ sampling}+L_{token\text{-}level}+L_{overlong}$$

DAPO는 하나의 마법 공식이 아니라 asymmetric clipping, informative group만 남기는 dynamic sampling, token-level loss, overlong shaping의 묶음입니다. Dr.GRPO는 normalization을 단순화하고 GSPO는 sequence-level ratio로 긴 응답의 token 변동을 모읍니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** reward가 `[0,0]` 또는 `[1,1]`인 group은 dynamic sampling에서 남을까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>아니요. group 내부 비교 신호가 없으므로 `[0,1]`, `[1,0]`만 남습니다.</details>

In [2]:
from rl_study.algorithms.dapo import dynamic_sampling_filter, overlong_reward_shaping
from rl_study.algorithms.grpo import dr_grpo_advantages, gspo_sequence_loss
candidate_rewards = torch.tensor([[0., 0.], [0., 1.], [1., 1.], [1., 0.]])
dynamic = dynamic_sampling_filter(candidate_rewards, required_groups=2)
penalties = overlong_reward_shaping(
    torch.tensor([5, 6, 8, 10]), max_response_length=10, buffer_length=4
)
current = torch.log(torch.tensor([[2., 8.], [1., 1.]]))
mask = torch.ones_like(current, dtype=torch.bool)
gspo = gspo_sequence_loss(
    current, torch.zeros_like(current), torch.zeros_like(current),
    torch.tensor([1., -1.]), mask, clip_low=10., clip_high=10.
)
print({"dynamic_indices": dynamic.selected_group_indices.tolist(),
       "overlong_penalty": penalties.tolist(),
       "dr_adv": dr_grpo_advantages(candidate_rewards[1:2]).tolist(),
       "gspo_ratio": gspo.ratio.tolist()})

{'dynamic_indices': [1, 3], 'overlong_penalty': [-0.0, -0.0, -0.5, -1.0], 'dr_adv': [[-0.5, 0.5]], 'gspo_ratio': [4.0, 1.0]}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** 각 변형을 독립 함수로 두면 어떤 안정화 요소가 결과를 바꿨는지 ablation할 수 있습니다. 논문 recipe를 이름 하나로 뭉치면 reduction과 mask 차이가 사라집니다.

**흔한 함정:** overlong penalty의 buffer 경계를 off-by-one으로 구현하면 최대 길이 직전부터 갑자기 -1이 됩니다. 시작·중간·끝 네 지점을 analytic test로 고정합니다. 회귀 test: `test_overlong_reward_shaping_boundaries`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert dynamic.selected_group_indices.tolist() == [1, 3]
assert penalties.tolist() == [0.0, 0.0, -0.5, -1.0]
print("checks=passed")

checks=passed


**회상 문제:** DAPO의 네 요소 중 sample 효율을 직접 겨냥하는 요소는 무엇인가요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** dynamic sampling은 group 1과 3만 골랐고 overlong penalty는 0→-0.5→-1로 변했습니다. GSPO ratio는 token ratio가 아니라 sequence 집계값입니다.
- 실제 확인: `test_overlong_reward_shaping_boundaries`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L14에서 같은 API를 공개 소형 모델과 GPU 서버로 옮길 때 다운로드·메모리·framework 경계를 확인합니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

[상세 구현 문서](../../docs/algorithms/dapo.md) · [강좌 지도](../../docs/course-map.md)

## Sources

- `dapo-2025` — `docs/sources.yml`
- `dr-grpo-2025` — `docs/sources.yml`
- `gspo-2025` — `docs/sources.yml`
- `repo-understand-r1-zero` — `docs/sources.yml`
- `framework-verl` — `docs/sources.yml`